# 📊 4주차 실습 — 데이터 전처리 + 기초 시각화

**과목**: 의료정보융합실무 | **담당**: 보건의료정보학과 장진수 교수  
**데이터**: 대학생 건강설문 (가상, 50명)

---

## 🗺️ 오늘 실습 흐름

| 단계 | 내용 | 강의 대응 |
|:---:|---|---|
| **STEP 1** | 데이터 불러오기 + 첫 확인 | Q7·Q8 |
| **STEP 2** | 결측치·이상치 탐색 | Q9 |
| **STEP 3** | 결측치 처리 (dropna / fillna) | Q10 |
| **STEP 4** | 이상치 처리 (조건 필터링) | Q10 |
| **STEP 5** | 히스토그램 — 분포 확인 | 시각화 실습 |
| **STEP 6** | 산점도 — 두 변수 관계 | 시각화 실습 |
| **STEP 7** | 막대그래프 — 범주별 비교 | 시각화 실습 |
| **STEP 8** | ✏️ 직접 해보기 | 응용 과제 |

> 💡 **코드를 외울 필요 없습니다** — 변수명만 바꾸면 여러분 조의 데이터에도 그대로 쓸 수 있습니다

---
## ⚙️ STEP 0 · 환경 준비 — 한글 폰트 설치

> Colab은 기본적으로 한글 폰트가 없습니다.  
> 아래 셀을 **처음 한 번만** 실행하세요. (약 10~20초 소요)

In [ ]:
# 한글 폰트 설치 (Colab 전용 — 처음 1회만)
!apt-get install -y -q fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fe = fm.FontEntry(fname='/usr/share/fonts/truetype/nanum/NanumGothic.ttf', name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams.update({
    'font.family'       : 'NanumGothic',
    'axes.unicode_minus': False,
    'figure.dpi'        : 110
})

print('✅ 한글 폰트 설정 완료')

---
## 📥 STEP 1-A · 데이터 불러오기

GitHub에 업로드된 CSV 파일을 URL로 바로 읽어옵니다.  
설치나 업로드 없이 한 줄로 불러올 수 있습니다.

> ⚠️ 아래 `RAW_URL` 은 교수님이 GitHub에 올린 뒤 주소로 바꾸세요

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── GitHub Raw URL로 바로 불러오기 ────────────────────────────
# 형식: https://raw.githubusercontent.com/계정명/저장소명/main/파일명.csv
RAW_URL = 'https://github.com/runmc3812/Healthcare_Informatics_Integration-2026-/blob/main/4%EC%A3%BC%EC%B0%A8%20%EC%8B%A4%EC%8A%B5/health_survey.csv'

df = pd.read_csv(RAW_URL)

print(f'✅ 데이터 불러오기 성공!')
print(f'   → {df.shape[0]}행 × {df.shape[1]}열')

#### 📎 GitHub URL을 모를 때 — 파일 직접 업로드 방법

```python
from google.colab import files
uploaded = files.upload()        # health_survey.csv 선택
df = pd.read_csv('health_survey.csv')
```

---
## 👁️ STEP 1-B · 데이터 첫 확인 — `df.head()`

> **엑셀로 치면**: 파일을 열어 처음 몇 줄 훑어보는 것  
> `df.head()` = 상위 5행 / `df.head(10)` = 상위 10행

In [ ]:
# 상위 5행 확인
df.head()

In [ ]:
# 하위 5행도 확인 (끝부분에 이상한 값이 숨어있을 수 있음)
df.tail()

---
## 🔍 STEP 2-A · 결측치 탐색 — `df.info()`

> **엑셀로 치면**: Ctrl+G → 빈 셀 선택  
> `non-null` 수가 50보다 작으면 → 그 열에 **결측치 존재**

In [ ]:
# 데이터 타입 + 결측치 개수 한 번에 확인
df.info()

In [ ]:
# 결측치 개수만 따로 뽑기
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(1)

missing_summary = pd.DataFrame({
    '결측치 수': missing,
    '결측 비율(%)': missing_pct
})

# 결측치가 있는 열만 출력
print('=== 결측치가 있는 열 ===')
print(missing_summary[missing_summary['결측치 수'] > 0])

---
## 🔍 STEP 2-B · 이상치 탐색 — `df.describe()`

> **엑셀로 치면**: 필터 → 오름차순/내림차순 정렬로 극단값 확인  
> `max` 값이 비정상적으로 크면 → **이상치 신호** ⚠️

In [ ]:
# 기술 통계 요약 (count·mean·std·min·max)
df.describe().round(2)

In [ ]:
# ⚠️ 이상치 체크리스트 — 각 변수의 정상 범위
print('=== 이상치 의심 값 탐색 ===')
print()

checks = [
    ('나이',       '나이 < 18 또는 나이 > 40', df[(df['나이'] < 18) | (df['나이'] > 40)]['나이']),
    ('수면시간(h)', '수면시간 > 24',            df[df['수면시간(h)'] > 24]['수면시간(h)']),
    ('PHQ9점수',   'PHQ9 < 0 또는 PHQ9 > 27',  df[(df['PHQ9점수'] < 0) | (df['PHQ9점수'] > 27)]['PHQ9점수']),
    ('스트레스점수','스트레스 > 40',             df[df['스트레스점수'] > 40]['스트레스점수']),
    ('BMI',        'BMI < 10 또는 BMI > 50',    df[(df['BMI'] < 10) | (df['BMI'] > 50)]['BMI']),
]

found = False
for col, condition, result in checks:
    if len(result) > 0:
        print(f'  ⚠️  [{col}] {condition}')
        # 해당 행의 ID 출력
        ids = df.loc[result.index, 'ID'].tolist()
        print(f'      해당 ID: {ids}  /  값: {result.tolist()}')
        found = True

if not found:
    print('  ✅ 이상치 없음')

---
## 🔧 STEP 3 · 결측치 처리 — `dropna` vs `fillna`

| 방법 | 코드 | 언제 쓰나 |
|---|---|---|
| **행 삭제** | `df.dropna()` | 결측치 비율이 낮을 때 |
| **평균값 대체** | `df['열'].fillna(df['열'].mean())` | 이상치가 없을 때 |
| **중앙값 대체** | `df['열'].fillna(df['열'].median())` | 이상치가 있을 때 (더 안전) |

> 💡 **원본은 항상 보존** — `df_clean` 이라는 새 변수에 저장합니다

In [ ]:
# 원본 보존 — 복사본에서 작업
df_clean = df.copy()

print('처리 전 결측치:')
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print()

In [ ]:
# ── 방법 1: 연속형 변수 → 중앙값으로 대체 ────────────────────
# BMI, 스마트폰(h), 카페인(mg) 결측치 → 중앙값 대체
for col in ['BMI', '스마트폰(h)', '카페인(mg)']:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f'  [{col}] 결측치 → 중앙값 {median_val:.1f} 으로 대체')

print()

# ── 방법 2: 범주형 변수 → 행 삭제 ────────────────────────────
# 성별 결측치 → 행 삭제 (2행 이하라면 삭제가 안전)
before = len(df_clean)
df_clean = df_clean.dropna(subset=['성별'])
after  = len(df_clean)
print(f'  [성별] 결측치 행 삭제: {before}행 → {after}행 ({before-after}행 제거)')

In [ ]:
# 처리 후 결측치 확인
remaining = df_clean.isnull().sum().sum()
print(f'처리 후 남은 결측치: {remaining}개')
if remaining == 0:
    print('✅ 결측치 처리 완료!')
else:
    print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

---
## 🔧 STEP 4 · 이상치 처리 — 조건 필터링

> **엑셀로 치면**: 필터 → 비정상 행 선택 후 삭제  
> Python에서는 **정상 조건을 만족하는 행만 남기는** 방식으로 처리합니다

In [ ]:
before = len(df_clean)

# 모든 이상치 조건을 한 번에 적용 (& = AND)
df_clean = df_clean[
    (df_clean['나이']       >= 18) & (df_clean['나이']       <= 40) &   # 나이 정상 범위
    (df_clean['수면시간(h)'] >= 3)  & (df_clean['수면시간(h)'] <= 24) &   # 수면 24시간 초과 제거
    (df_clean['PHQ9점수']   >= 0)  & (df_clean['PHQ9점수']   <= 27) &   # PHQ9 범위 0~27
    (df_clean['스트레스점수'] >= 0) & (df_clean['스트레스점수'] <= 40)    # 스트레스 0~40
]

after = len(df_clean)
print(f'이상치 처리: {before}행 → {after}행 ({before-after}행 제거)')
print(f'\n최종 분석 데이터: {df_clean.shape[0]}명 × {df_clean.shape[1]}개 변수')

In [ ]:
# 전처리 전·후 기술통계 비교
compare_cols = ['나이', '수면시간(h)', 'BMI', 'PHQ9점수', '스트레스점수']

before_stats = df[compare_cols].describe().loc[['mean','min','max']].round(2)
after_stats  = df_clean[compare_cols].describe().loc[['mean','min','max']].round(2)

print('=== 전처리 전 ===')
print(before_stats)
print('\n=== 전처리 후 ===')
print(after_stats)

---
## 📊 STEP 5 · 히스토그램 — 분포 확인

> **언제 쓰나**: 연속형 변수의 분포 형태 파악  
> **질문 예시**: "수면시간이 주로 몇 시간에 집중되는가?"

In [ ]:
# ── 기본 히스토그램 (강의 예시 코드) ─────────────────────────
plt.hist(df_clean['수면시간(h)'], bins=10, color='steelblue', edgecolor='white')
plt.title('수면시간 분포')
plt.xlabel('수면시간 (h)')
plt.ylabel('인원 수')
plt.show()

In [ ]:
# ── 주요 연속형 변수 4개 히스토그램 한 번에 보기 ─────────────
cont_vars = ['수면시간(h)', 'BMI', 'PHQ9점수', '스트레스점수']
colors    = ['#5B8DB8', '#2E7D91', '#C84B31', '#E09F3E']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (var, color) in enumerate(zip(cont_vars, colors)):
    ax = axes[i]
    ax.hist(df_clean[var], bins=12, color=color, edgecolor='white', alpha=0.85)

    # 평균선 추가
    mean_val = df_clean[var].mean()
    ax.axvline(mean_val, color='black', linewidth=1.5, linestyle='--',
               label=f'평균 {mean_val:.1f}')

    ax.set_title(f'{var} 분포', fontsize=12, fontweight='bold')
    ax.set_xlabel(var)
    ax.set_ylabel('인원 수')
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

fig.suptitle('주요 연속형 변수 분포', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('histogram.png', bbox_inches='tight')  # 저장
plt.show()
print('저장: histogram.png')

---
## 🔵 STEP 6 · 산점도 — 두 변수의 관계 확인

> **언제 쓰나**: 두 연속형 변수 사이의 관계 파악  
> **질문 예시**: "수면시간이 짧을수록 우울감(PHQ9)이 높은가?"

In [ ]:
# ── 기본 산점도 (강의 예시 코드) ──────────────────────────────
plt.scatter(df_clean['수면시간(h)'], df_clean['PHQ9점수'])
plt.title('수면시간 vs 우울 척도')
plt.xlabel('수면시간 (h)')
plt.ylabel('PHQ9 점수')
plt.show()

In [ ]:
# ── 성별로 색 구분 + 추세선 추가 산점도 ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: 수면시간 vs PHQ9
ax = axes[0]
colors_gender = {'남': '#5B8DB8', '여': '#C84B31'}
for gender, color in colors_gender.items():
    sub = df_clean[df_clean['성별'] == gender]
    ax.scatter(sub['수면시간(h)'], sub['PHQ9점수'],
               color=color, alpha=0.7, label=gender, s=60)

# 전체 추세선
z = np.polyfit(df_clean['수면시간(h)'], df_clean['PHQ9점수'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_clean['수면시간(h)'].min(), df_clean['수면시간(h)'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, alpha=0.6, label='추세선')

# 상관계수 표시
r = df_clean['수면시간(h)'].corr(df_clean['PHQ9점수'])
ax.set_title(f'수면시간 vs PHQ9점수 (r = {r:.2f})', fontsize=12, fontweight='bold')
ax.set_xlabel('수면시간 (h)')
ax.set_ylabel('PHQ9 점수')
ax.legend()
sns.despine(ax=ax)

# 오른쪽: 스마트폰 vs 스트레스
ax = axes[1]
for gender, color in colors_gender.items():
    sub = df_clean[df_clean['성별'] == gender]
    ax.scatter(sub['스마트폰(h)'], sub['스트레스점수'],
               color=color, alpha=0.7, label=gender, s=60)

z2 = np.polyfit(df_clean['스마트폰(h)'], df_clean['스트레스점수'], 1)
p2 = np.poly1d(z2)
x2 = np.linspace(df_clean['스마트폰(h)'].min(), df_clean['스마트폰(h)'].max(), 100)
ax.plot(x2, p2(x2), 'k--', linewidth=1.5, alpha=0.6, label='추세선')

r2 = df_clean['스마트폰(h)'].corr(df_clean['스트레스점수'])
ax.set_title(f'스마트폰 사용 vs 스트레스 (r = {r2:.2f})', fontsize=12, fontweight='bold')
ax.set_xlabel('스마트폰 사용 시간 (h)')
ax.set_ylabel('스트레스 점수')
ax.legend()
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('scatter.png', bbox_inches='tight')
plt.show()
print('저장: scatter.png')

---
## 📈 STEP 7 · 막대그래프 — 범주별 평균 비교

> **언제 쓰나**: 범주형 변수에 따라 수치가 어떻게 다른지 비교  
> **질문 예시**: "흡연 여부에 따라 스트레스 점수가 다른가?"

In [ ]:
# ── 성별에 따른 수면시간 평균 비교 ────────────────────────────
sleep_by_gender = df_clean.groupby('성별')['수면시간(h)'].mean().round(2)
print('성별별 평균 수면시간:')
print(sleep_by_gender)

plt.figure(figsize=(5, 4))
bars = plt.bar(sleep_by_gender.index, sleep_by_gender.values,
               color=['#5B8DB8', '#C84B31'], edgecolor='white', width=0.5)

# 막대 위에 수치 표시
for bar, val in zip(bars, sleep_by_gender.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}h', ha='center', va='bottom', fontsize=11)

plt.title('성별에 따른 평균 수면시간', fontsize=12, fontweight='bold')
plt.xlabel('성별')
plt.ylabel('평균 수면시간 (h)')
plt.ylim(0, sleep_by_gender.max() * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── 범주형 변수 4개 한 번에 비교 ─────────────────────────────
cat_vars  = ['성별', '흡연여부', '운동여부', '음주여부']
target    = 'PHQ9점수'   # 비교할 수치 변수

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
palette   = ['#5B8DB8', '#C84B31', '#2E7D91', '#E09F3E']

for ax, cat, color in zip(axes, cat_vars, palette):
    group_means = df_clean.groupby(cat)[target].mean().round(2)
    bars = ax.bar(group_means.index, group_means.values,
                  color=color, edgecolor='white', alpha=0.85, width=0.5)

    for bar, val in zip(bars, group_means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)

    ax.set_title(f'{cat}별 {target}', fontsize=11, fontweight='bold')
    ax.set_xlabel(cat)
    ax.set_ylabel(target if ax == axes[0] else '')
    ax.set_ylim(0, group_means.max() * 1.3)
    sns.despine(ax=ax)

fig.suptitle(f'범주형 변수에 따른 {target} 비교', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bar_comparison.png', bbox_inches='tight')
plt.show()
print('저장: bar_comparison.png')

---
## 📋 STEP 7-B · 기초통계 요약표 출력

> 강의 미션 5 — 기초통계 계산 결과를 표로 정리합니다

In [ ]:
# 연속형 변수 기초통계 요약
cont_cols = ['나이', '수면시간(h)', 'BMI', 'PHQ9점수', '스마트폰(h)', '스트레스점수', '카페인(mg)']

stats = df_clean[cont_cols].agg(['mean','median','std','min','max']).T.round(2)
stats.columns = ['평균', '중앙값', '표준편차', '최솟값', '최댓값']
print('=== 연속형 변수 기초통계 요약 ===')
print(stats)
stats

In [ ]:
# 범주형 변수 빈도표
print('=== 범주형 변수 빈도 ===')
for col in ['성별', '흡연여부', '운동여부', '음주여부', '학년', '거주지역']:
    counts = df_clean[col].value_counts()
    pct    = df_clean[col].value_counts(normalize=True).map('{:.1%}'.format)
    tbl    = pd.DataFrame({'n': counts, '%': pct})
    print(f'\n[{col}]')
    print(tbl.to_string())

---
## ✏️ STEP 8 · 직접 해보기 — 응용 과제

아래 셀의 `?` 부분을 채워서 코드를 완성해 보세요!

### 과제 1 · 히스토그램 변수 바꾸기
> `수면시간(h)` 대신 **BMI** 분포를 히스토그램으로 그려보세요

In [ ]:
# 과제 1: ? 자리를 채워보세요
plt.hist(df_clean['?'], bins=10, color='steelblue', edgecolor='white')
plt.title('? 분포')
plt.xlabel('?')
plt.ylabel('인원 수')
plt.show()

### 과제 2 · 산점도 변수 바꾸기
> **카페인 섭취량** vs **스트레스 점수** 산점도를 그려보세요  
> 상관계수(r)도 같이 출력해보세요

In [ ]:
# 과제 2: ? 자리를 채워보세요
r = df_clean['?'].corr(df_clean['?'])
print(f'상관계수 r = {r:.2f}')

plt.scatter(df_clean['?'], df_clean['?'], alpha=0.7, color='steelblue')
plt.title(f'? vs ? (r = {r:.2f})')
plt.xlabel('?')
plt.ylabel('?')
plt.show()

### 과제 3 · 그룹 비교
> **운동 여부**에 따라 **PHQ9 점수** 평균이 다른지 막대그래프로 비교해보세요

In [ ]:
# 과제 3: ? 자리를 채워보세요
group_means = df_clean.groupby('?')['?'].mean().round(2)
print(group_means)

plt.bar(group_means.index, group_means.values, color=['#5B8DB8','#C84B31'], width=0.5)
plt.title('? 여부에 따른 ? 비교')
plt.xlabel('?')
plt.ylabel('평균 ?')
plt.show()

---
## ✅ 오늘 배운 것 정리

| 코드 | 엑셀에서 동일 작업 | 용도 |
|---|---|---|
| `df.head()` | 파일 열어 첫 줄 확인 | 데이터 미리보기 |
| `df.info()` | Ctrl+G → 빈 셀 | 결측치 + 데이터 타입 확인 |
| `df.describe()` | 정렬 후 최대/최소 확인 | 이상치 탐지 |
| `df.dropna()` | 행 선택 후 삭제 | 결측치 제거 |
| `df['열'].fillna(중앙값)` | MEDIAN 값 입력 | 결측치 대체 |
| `df[df['열'] <= 값]` | 필터 → 조건 설정 | 이상치 제거 |
| `plt.hist()` | 차트 → 히스토그램 | 분포 확인 |
| `plt.scatter()` | 차트 → 분산형 | 두 변수 관계 |
| `plt.bar()` | 차트 → 막대 | 범주별 비교 |

---

---
*의료정보융합실무 | 4주차 | 보건의료정보학과 장진수 교수*